# 📊 Minería de Datos — Regresión Supervisada
## Clase magistral completa para dictar — 2 horas

**Asignatura:** Minería de Datos  
**Tema:** Regresión lineal, regularización y árboles de regresión  
**Duración estimada:** 2 horas  
**Nivel:** Ingeniería de Sistemas  

---

# 🎯 Propósito de la clase

En las clases anteriores hemos trabajado modelos de **clasificación**, donde el objetivo era predecir una categoría:

- Sobrevive / no sobrevive
- Enfermo / sano
- Fraude / no fraude
- Compra / no compra

Hoy cambiamos de paradigma.

En esta clase ya no queremos predecir una clase, sino un **valor numérico continuo**.

---

# Frase de apertura para el docente

> “Hasta ahora nuestros modelos respondían preguntas del tipo: ¿a qué clase pertenece este registro?  
> Hoy vamos a responder una pregunta diferente: ¿cuánto vale, cuánto cuesta, cuánto tarda o cuánto se espera?”

---

# Resultado esperado de aprendizaje

Al finalizar esta clase, el estudiante debe poder:

1. Diferenciar clasificación y regresión.
2. Formular un problema de regresión supervisada.
3. Entrenar una regresión lineal.
4. Interpretar coeficientes.
5. Evaluar modelos de regresión con MSE, RMSE, MAE y R².
6. Comprender overfitting y underfitting en regresión.
7. Comparar regresión lineal, Ridge, Lasso y árbol de regresión.
8. Justificar qué modelo usaría según desempeño e interpretabilidad.

# 1. Clasificación vs Regresión

Antes de entrar en código, dejemos clara la diferencia conceptual.

| Aspecto | Clasificación | Regresión |
|---|---|---|
| Variable objetivo | Categórica | Numérica continua |
| Ejemplo | Sobrevive / no sobrevive | Precio de vivienda |
| Salida del modelo | Clase | Valor |
| Evaluación | Accuracy, Precision, Recall, F1 | MAE, MSE, RMSE, R² |
| Error típico | Clase incorrecta | Diferencia entre valor real y predicho |

---

# Explicación docente

En clasificación, un error puede ser decir que alguien sobrevivió cuando realmente no sobrevivió.

En regresión, el error es numérico.  
Por ejemplo, si una casa vale 300.000 dólares y el modelo predice 280.000, el error es 20.000.

---

# Pregunta para estudiantes

> Si quiero predecir si un cliente se va o se queda, ¿es clasificación o regresión?

> Si quiero predecir cuánto dinero gastará un cliente el próximo mes, ¿es clasificación o regresión?

# 2. Problema de la clase

Trabajaremos con un problema clásico:

> Predecir el valor medio de viviendas en California a partir de características socioeconómicas y geográficas.

Usaremos el dataset **California Housing**, disponible en `scikit-learn`.

---

# Variable objetivo

La variable objetivo será:

\[
MedHouseVal
\]

Esta representa el valor medio de vivienda en una zona.

---

# Variables predictoras

Algunas variables del dataset son:

| Variable | Significado |
|---|---|
| MedInc | Ingreso medio de la zona |
| HouseAge | Edad media de las viviendas |
| AveRooms | Promedio de habitaciones |
| AveBedrms | Promedio de dormitorios |
| Population | Población |
| AveOccup | Ocupación promedio |
| Latitude | Latitud |
| Longitude | Longitud |

---

# Frase docente

> “Este dataset nos permite conectar minería de datos con problemas reales de valoración, planeación urbana, mercado inmobiliario y análisis socioeconómico.”

In [ ]:
# ============================================================
# 1. Importar librerías
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

# 3. Carga del dataset

Ahora cargamos el dataset.

Este dataset ya viene preparado en `scikit-learn`, pero representa un problema real de regresión.

---

# Nota docente

Aunque no lo descargamos desde una URL externa, este dataset es ideal para clase porque:

1. No depende de internet.
2. Está documentado.
3. Tiene variables numéricas.
4. Permite comparar varios modelos.
5. Permite explicar errores de regresión con claridad.

In [ ]:
# ============================================================
# 2. Cargar dataset California Housing
# ============================================================

data = fetch_california_housing(as_frame=True)

df = data.frame

print("Dimensiones del dataset:", df.shape)
df.head()

# 4. Comprensión inicial de los datos

Antes de modelar debemos conocer los datos.

Preguntas mínimas:

1. ¿Cuántas observaciones hay?
2. ¿Cuántas variables predictoras?
3. ¿Cuál es la variable objetivo?
4. ¿Hay valores faltantes?
5. ¿Qué escala tienen las variables?
6. ¿La variable objetivo parece tener valores extremos?

Este paso corresponde a la fase de comprensión de datos dentro de CRISP-DM.

In [ ]:
# Información general
df.info()

In [ ]:
# Valores faltantes
df.isna().sum()

In [ ]:
# Resumen estadístico
df.describe().T

# Interpretación docente

Observe que todas las variables son numéricas.  
Esto facilita el uso de modelos como regresión lineal, Ridge, Lasso y árboles.

Pero también observe que las escalas son diferentes:

- `Population` puede tener valores grandes.
- `Latitude` y `Longitude` tienen rangos distintos.
- `MedInc` tiene otra escala.

Esto será importante cuando usemos modelos regularizados, porque Ridge y Lasso son sensibles a la escala de las variables.

# 5. Exploración de la variable objetivo

La variable objetivo es `MedHouseVal`.

Antes de entrenar modelos, revisemos su distribución.

In [ ]:
plt.figure(figsize=(8,4))
plt.hist(df["MedHouseVal"], bins=40)
plt.title("Distribución de la variable objetivo: MedHouseVal")
plt.xlabel("MedHouseVal")
plt.ylabel("Frecuencia")
plt.show()

# Preguntas para clase

1. ¿La variable objetivo parece uniforme?
2. ¿Hay concentración de valores en algún rango?
3. ¿Existen posibles valores extremos?
4. ¿Qué puede pasar si el modelo intenta predecir valores muy altos?

---

# Frase docente

> “Antes de medir si un modelo predice bien, debo entender qué está intentando predecir.”

# 6. Relación entre variables predictoras y objetivo

Revisemos algunas relaciones simples.

No buscamos hacer un análisis estadístico completo, sino observar si ciertas variables parecen asociarse con el valor de vivienda.

In [ ]:
variables_a_graficar = ["MedInc", "HouseAge", "AveRooms", "Population"]

for var in variables_a_graficar:
    plt.figure(figsize=(6,4))
    plt.scatter(df[var], df["MedHouseVal"], alpha=0.2)
    plt.title(f"{var} vs MedHouseVal")
    plt.xlabel(var)
    plt.ylabel("MedHouseVal")
    plt.show()

# Interpretación docente

La variable `MedInc` suele mostrar una relación positiva con el valor de vivienda.

Esto tiene sentido:

> Zonas con mayor ingreso medio tienden a tener viviendas de mayor valor.

Pero recuerde:

**correlación no implica causalidad**.

Un modelo predictivo puede usar una variable para mejorar predicción sin afirmar que sea la causa directa.

# 7. Separación entre X e y

Definimos:

\[
X = \text{variables predictoras}
\]

\[
y = \text{variable objetivo}
\]

En este caso:

- `X`: todas las columnas excepto `MedHouseVal`
- `y`: `MedHouseVal`

In [ ]:
X = df.drop(columns=["MedHouseVal"])
y = df["MedHouseVal"]

print("X:", X.shape)
print("y:", y.shape)
print("Variables predictoras:")
print(list(X.columns))

# 8. Train/Test split

Separaremos:

- 70% entrenamiento
- 30% prueba

---

# Explicación docente

El modelo aprende con entrenamiento.

El conjunto de prueba representa datos no vistos.

Si evaluamos solo en entrenamiento, podemos engañarnos porque el modelo pudo memorizar patrones específicos.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

# 9. Métricas de regresión

En clasificación usábamos accuracy, precision, recall y F1.

En regresión usamos métricas basadas en error.

---

## 9.1 Error individual

Para cada observación:

\[
e_i = y_i - \hat{y}_i
\]

Donde:

- \(y_i\): valor real.
- \(\hat{y}_i\): valor predicho.
- \(e_i\): error.

---

## 9.2 MAE — Mean Absolute Error

\[
MAE = \frac{1}{n}\sum_{i=1}^{n}|y_i - \hat{y}_i|
\]

Interpretación:

> Error promedio absoluto.

Es fácil de interpretar porque está en la misma escala de la variable objetivo.

---

## 9.3 MSE — Mean Squared Error

\[
MSE = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2
\]

Penaliza más los errores grandes.

---

## 9.4 RMSE — Root Mean Squared Error

\[
RMSE = \sqrt{MSE}
\]

Está en la escala original de la variable objetivo.

---

## 9.5 R²

\[
R^2 = 1 - \frac{SS_{res}}{SS_{tot}}
\]

Interpretación:

> Proporción de variabilidad de la variable objetivo explicada por el modelo.

Un R² cercano a 1 indica mejor ajuste.

In [ ]:
# ============================================================
# 3. Función de evaluación
# ============================================================

def evaluar_regresion(nombre, y_real, y_pred):
    mae = mean_absolute_error(y_real, y_pred)
    mse = mean_squared_error(y_real, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_real, y_pred)

    return {
        "modelo": nombre,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    }

# 10. Regresión lineal

La regresión lineal modela la relación entre variables predictoras y una variable objetivo continua.

La forma general es:

\[
\hat{y} = \beta_0 + \beta_1x_1 + \beta_2x_2 + ... + \beta_px_p
\]

Donde:

- \(\hat{y}\): predicción.
- \(\beta_0\): intercepto.
- \(\beta_j\): coeficiente de la variable \(x_j\).
- \(p\): número de variables.

---

# Interpretación

Si \(\beta_j\) es positivo:

> Al aumentar \(x_j\), la predicción tiende a aumentar, manteniendo las demás variables constantes.

Si \(\beta_j\) es negativo:

> Al aumentar \(x_j\), la predicción tiende a disminuir, manteniendo las demás variables constantes.

In [ ]:
# ============================================================
# 4. Modelo de regresión lineal
# ============================================================

modelo_lineal = LinearRegression()

modelo_lineal.fit(X_train, y_train)

pred_lineal = modelo_lineal.predict(X_test)

evaluar_regresion("Regresión lineal", y_test, pred_lineal)

# 11. Interpretación de coeficientes

Los coeficientes permiten analizar cómo cada variable afecta la predicción dentro del modelo lineal.

---

# Advertencia docente

La interpretación de coeficientes debe hacerse con cuidado:

1. Depende de la escala de las variables.
2. No implica causalidad.
3. Puede verse afectada por multicolinealidad.
4. Supone relación lineal.

In [ ]:
coeficientes = pd.DataFrame({
    "variable": X.columns,
    "coeficiente": modelo_lineal.coef_
}).sort_values("coeficiente", ascending=False)

coeficientes

In [ ]:
plt.figure(figsize=(9,5))
plt.barh(coeficientes["variable"], coeficientes["coeficiente"])
plt.title("Coeficientes de la regresión lineal")
plt.xlabel("Coeficiente")
plt.ylabel("Variable")
plt.axvline(0)
plt.show()

# Pregunta para estudiantes

> ¿Cuál variable tiene el coeficiente positivo más alto?

> ¿Cuál tiene coeficiente negativo?

> ¿Podemos decir que esa variable causa el precio de vivienda?

La respuesta esperada es: **no necesariamente**.  
El modelo muestra asociación predictiva, no causalidad.

# 12. Diagnóstico visual: real vs predicho

Una forma sencilla de evaluar un modelo de regresión es graficar:

- Valores reales.
- Valores predichos.

Si el modelo fuera perfecto, los puntos estarían cerca de una línea diagonal.

In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(y_test, pred_lineal, alpha=0.3)
plt.xlabel("Valor real")
plt.ylabel("Valor predicho")
plt.title("Regresión lineal: Real vs Predicho")

min_val = min(y_test.min(), pred_lineal.min())
max_val = max(y_test.max(), pred_lineal.max())
plt.plot([min_val, max_val], [min_val, max_val])

plt.show()

# Interpretación docente

Si los puntos están muy dispersos, el modelo tiene errores importantes.

Si se concentran cerca de la diagonal, el modelo predice mejor.

Este gráfico permite ver cosas que una métrica sola no muestra.

# 13. Análisis de residuos

El residuo es:

\[
residuo = y - \hat{y}
\]

Un residuo positivo significa que el modelo predijo por debajo del valor real.

Un residuo negativo significa que el modelo predijo por encima del valor real.

In [ ]:
residuos_lineal = y_test - pred_lineal

plt.figure(figsize=(8,4))
plt.hist(residuos_lineal, bins=40)
plt.title("Distribución de residuos - Regresión lineal")
plt.xlabel("Residuo")
plt.ylabel("Frecuencia")
plt.show()

In [ ]:
plt.figure(figsize=(7,4))
plt.scatter(pred_lineal, residuos_lineal, alpha=0.3)
plt.axhline(0)
plt.title("Residuos vs Predicción")
plt.xlabel("Predicción")
plt.ylabel("Residuo")
plt.show()

# Preguntas de interpretación

1. ¿Los residuos están centrados alrededor de cero?
2. ¿Hay patrones visibles?
3. ¿Los errores aumentan para valores altos?
4. ¿El modelo parece tener dificultad en algún rango?

---

# Frase docente

> “Las métricas resumen el error, pero los gráficos muestran la forma del error.”

# 14. Train vs Test: overfitting y underfitting

En regresión también existe overfitting.

## Underfitting

El modelo es demasiado simple y falla tanto en train como en test.

## Overfitting

El modelo aprende demasiado bien el entrenamiento, pero generaliza mal.

---

# Señal típica

| Caso | Train | Test |
|---|---|---|
| Buen ajuste | buen desempeño | buen desempeño |
| Underfitting | mal desempeño | mal desempeño |
| Overfitting | excelente desempeño | peor desempeño |

In [ ]:
pred_lineal_train = modelo_lineal.predict(X_train)

resultado_lineal_train = evaluar_regresion("Lineal train", y_train, pred_lineal_train)
resultado_lineal_test = evaluar_regresion("Lineal test", y_test, pred_lineal)

pd.DataFrame([resultado_lineal_train, resultado_lineal_test])

# 15. Ridge Regression

Ridge es una regresión lineal regularizada.

La idea es penalizar coeficientes demasiado grandes.

La función objetivo incluye un término de penalización:

\[
\sum (y_i - \hat{y}_i)^2 + \alpha \sum \beta_j^2
\]

Donde:

- \(\alpha\) controla la fuerza de la penalización.
- Si \(\alpha\) es muy pequeño, Ridge se parece a regresión lineal.
- Si \(\alpha\) es muy grande, los coeficientes se reducen demasiado.

---

# Explicación docente

> “Ridge no solo busca ajustar bien; también busca que el modelo no dependa excesivamente de coeficientes muy grandes.”

# ¿Por qué usamos escalado en Ridge?

Ridge penaliza coeficientes.

Si las variables están en escalas diferentes, la penalización no es justa.

Por eso usamos `StandardScaler` dentro de un `Pipeline`.

In [ ]:
# ============================================================
# 5. Ridge Regression
# ============================================================

modelo_ridge = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=1.0))
])

modelo_ridge.fit(X_train, y_train)

pred_ridge = modelo_ridge.predict(X_test)

evaluar_regresion("Ridge alpha=1", y_test, pred_ridge)

# 16. Lasso Regression

Lasso también es una regresión lineal regularizada, pero usa otra penalización:

\[
\sum (y_i - \hat{y}_i)^2 + \alpha \sum |\beta_j|
\]

La diferencia clave es que Lasso puede llevar algunos coeficientes exactamente a cero.

---

# Interpretación

Lasso puede funcionar como mecanismo de selección de variables.

Si una variable queda con coeficiente cero, el modelo la está descartando.

In [ ]:
# ============================================================
# 6. Lasso Regression
# ============================================================

modelo_lasso = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Lasso(alpha=0.01, max_iter=10000))
])

modelo_lasso.fit(X_train, y_train)

pred_lasso = modelo_lasso.predict(X_test)

evaluar_regresion("Lasso alpha=0.01", y_test, pred_lasso)

In [ ]:
# Coeficientes de Lasso
lasso_coef = modelo_lasso.named_steps["model"].coef_

coef_lasso = pd.DataFrame({
    "variable": X.columns,
    "coeficiente_lasso": lasso_coef
}).sort_values("coeficiente_lasso", ascending=False)

coef_lasso

# Pregunta para estudiantes

> ¿Lasso eliminó alguna variable?

Si algún coeficiente es cero o muy cercano a cero, se puede discutir que el modelo redujo la importancia de esa variable.

---

# Nota docente

Lasso no “entiende” el mundo.  
Solo penaliza matemáticamente coeficientes y puede dejar algunos en cero.

# 17. Comparación de alpha en Ridge y Lasso

El parámetro \(\alpha\) controla la fuerza de regularización.

Probemos varios valores.

In [ ]:
alphas = [0.001, 0.01, 0.1, 1, 10, 100]

resultados_alpha = []

for alpha in alphas:
    ridge = Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=alpha))
    ])
    ridge.fit(X_train, y_train)
    pred = ridge.predict(X_test)
    resultados_alpha.append(evaluar_regresion(f"Ridge alpha={alpha}", y_test, pred))

    lasso = Pipeline([
        ("scaler", StandardScaler()),
        ("model", Lasso(alpha=alpha, max_iter=10000))
    ])
    lasso.fit(X_train, y_train)
    pred = lasso.predict(X_test)
    resultados_alpha.append(evaluar_regresion(f"Lasso alpha={alpha}", y_test, pred))

df_alpha = pd.DataFrame(resultados_alpha)
df_alpha

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(df_alpha["modelo"], df_alpha["RMSE"], marker="o")
plt.title("Comparación de RMSE según modelo y alpha")
plt.xlabel("Modelo")
plt.ylabel("RMSE")
plt.xticks(rotation=75)
plt.grid(True)
plt.show()

# Interpretación docente

Al aumentar alpha:

- Ridge reduce coeficientes.
- Lasso puede eliminar variables.
- Si alpha es demasiado grande, el modelo puede subajustar.

---

# Pregunta para clase

> ¿Cuál alpha tuvo menor RMSE?

> ¿Cuál modelo parece más estable?

# 18. Árbol de regresión

Un árbol de regresión usa lógica similar al árbol de clasificación, pero cambia la salida.

En clasificación:

> La hoja predice una clase.

En regresión:

> La hoja predice un valor promedio.

---

# Ejemplo conceptual

```text
Si MedInc <= 3.5
    predicción = 1.2
Si MedInc > 3.5 y Latitude <= 35
    predicción = 2.8
```

Cada hoja contiene un valor numérico.

In [ ]:
# ============================================================
# 7. Árbol de regresión
# ============================================================

modelo_arbol = DecisionTreeRegressor(
    max_depth=5,
    random_state=42
)

modelo_arbol.fit(X_train, y_train)

pred_arbol = modelo_arbol.predict(X_test)

evaluar_regresion("Árbol max_depth=5", y_test, pred_arbol)

# 19. Visualización parcial del árbol

Visualizar un árbol completo puede ser pesado.

Mostraremos una versión limitada para interpretar la lógica.

In [ ]:
plt.figure(figsize=(22, 10))
plot_tree(
    modelo_arbol,
    feature_names=X.columns,
    filled=True,
    rounded=True,
    max_depth=3,
    fontsize=8
)
plt.title("Árbol de regresión — visualización parcial")
plt.show()

# 20. Profundidad del árbol y overfitting

Probemos varias profundidades.

Un árbol muy profundo puede memorizar el entrenamiento.

Compararemos RMSE en train y test.

In [ ]:
profundidades = [1, 2, 3, 4, 5, 8, 12, None]
resultados_arboles = []

for p in profundidades:
    arbol = DecisionTreeRegressor(
        max_depth=p,
        random_state=42
    )

    arbol.fit(X_train, y_train)

    pred_train = arbol.predict(X_train)
    pred_test = arbol.predict(X_test)

    resultados_arboles.append({
        "max_depth": str(p),
        "RMSE_train": np.sqrt(mean_squared_error(y_train, pred_train)),
        "RMSE_test": np.sqrt(mean_squared_error(y_test, pred_test)),
        "R2_test": r2_score(y_test, pred_test)
    })

df_arboles = pd.DataFrame(resultados_arboles)
df_arboles

In [ ]:
plt.figure(figsize=(9,5))
plt.plot(df_arboles["max_depth"], df_arboles["RMSE_train"], marker="o", label="Train")
plt.plot(df_arboles["max_depth"], df_arboles["RMSE_test"], marker="o", label="Test")
plt.title("Efecto de la profundidad del árbol en regresión")
plt.xlabel("max_depth")
plt.ylabel("RMSE")
plt.legend()
plt.grid(True)
plt.show()

# Interpretación esperada

Cuando la profundidad aumenta:

- El error en train suele bajar.
- El error en test puede bajar al inicio.
- Después puede subir o estabilizarse.
- Si train mejora mucho y test no, hay señales de overfitting.

---

# Frase docente

> “Un modelo más complejo no siempre predice mejor; a veces solo memoriza mejor.”

# 21. Comparación final de modelos

Comparemos:

1. Regresión lineal.
2. Ridge.
3. Lasso.
4. Árbol de regresión.

Usaremos MAE, RMSE y R².

In [ ]:
resultados_finales = pd.DataFrame([
    evaluar_regresion("Regresión lineal", y_test, pred_lineal),
    evaluar_regresion("Ridge alpha=1", y_test, pred_ridge),
    evaluar_regresion("Lasso alpha=0.01", y_test, pred_lasso),
    evaluar_regresion("Árbol max_depth=5", y_test, pred_arbol)
])

resultados_finales.sort_values("RMSE")

In [ ]:
plt.figure(figsize=(8,5))
plt.bar(resultados_finales["modelo"], resultados_finales["RMSE"])
plt.title("Comparación de RMSE por modelo")
plt.ylabel("RMSE")
plt.xticks(rotation=30)
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.bar(resultados_finales["modelo"], resultados_finales["R2"])
plt.title("Comparación de R² por modelo")
plt.ylabel("R²")
plt.xticks(rotation=30)
plt.show()

# 22. ¿Cuál modelo elegir?

La elección no depende solo de la métrica.

| Criterio | Modelo que puede convenir |
|---|---|
| Interpretación de coeficientes | Regresión lineal |
| Control de coeficientes | Ridge |
| Selección de variables | Lasso |
| Relaciones no lineales | Árbol de regresión |
| Explicación sencilla por reglas | Árbol |
| Estabilidad lineal | Ridge |

---

# Pregunta para estudiantes

> Si el gerente necesita entender claramente cómo cada variable afecta el precio, ¿qué modelo elegiría?

> Si solo interesa reducir error, ¿qué modelo parece mejor?

> Si el árbol tiene mejor RMSE pero es menos estable, ¿lo usaría directamente?

# 23. Errores comunes en regresión

| Error | Consecuencia |
|---|---|
| Evaluar solo con R² | Puede ocultar errores grandes |
| No revisar residuos | No se entiende dónde falla |
| No separar train/test | Evaluación engañosa |
| No escalar con Ridge/Lasso | Penalización injusta |
| Interpretar coeficientes como causalidad | Conclusiones falsas |
| Usar árbol demasiado profundo | Overfitting |
| Elegir modelo solo por métrica | Mala decisión técnica |

---

# Frase docente

> “En minería de datos, entrenar el modelo es solo una parte; interpretar el error es lo que convierte el modelo en conocimiento.”

# 🧪 24. Taller final de clase

## Objetivo

Aplicar modelos de regresión para predecir el valor medio de vivienda y comparar su desempeño.

## Forma de trabajo

Grupos de 2 o 3 estudiantes.

## Entregable

Un notebook o documento con:

1. Contexto del problema.
2. Separación X/y.
3. Train/test.
4. Regresión lineal.
5. Ridge y Lasso.
6. Árbol de regresión.
7. Comparación de métricas.
8. Gráficos de error.
9. Conclusión técnica.

---

No se califica solo el código.  
Se califica especialmente la interpretación.

## Parte A — Comprensión conceptual

Responda:

1. ¿Por qué este problema es de regresión y no de clasificación?
2. ¿Cuál es la variable objetivo?
3. ¿Qué representa una predicción del modelo?
4. ¿Qué significa cometer error en regresión?
5. ¿Qué diferencia hay entre MAE, RMSE y R²?

## Parte B — Regresión lineal

Entrene una regresión lineal y responda:

1. ¿Cuál es el RMSE?
2. ¿Cuál es el R²?
3. ¿Qué variable tiene mayor coeficiente positivo?
4. ¿Qué variable tiene coeficiente negativo?
5. ¿Se pueden interpretar estos coeficientes como causalidad? Justifique.

In [ ]:
# Parte B - Espacio de trabajo

modelo_b = LinearRegression()
modelo_b.fit(X_train, y_train)
pred_b = modelo_b.predict(X_test)

print(evaluar_regresion("Lineal taller", y_test, pred_b))

pd.DataFrame({
    "variable": X.columns,
    "coeficiente": modelo_b.coef_
}).sort_values("coeficiente", ascending=False)

## Parte C — Ridge y Lasso

Entrene Ridge y Lasso con tres valores de alpha:

- 0.01
- 1
- 100

Responda:

1. ¿Cómo cambia el RMSE?
2. ¿Cómo cambia R²?
3. ¿Qué pasa con Lasso cuando alpha aumenta?
4. ¿Cuál modelo regularizado elegiría?

In [ ]:
# Parte C - Espacio de trabajo

alphas_taller = [0.01, 1, 100]
resultados_c = []

for alpha in alphas_taller:
    ridge = Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=alpha))
    ])
    ridge.fit(X_train, y_train)
    pred = ridge.predict(X_test)
    resultados_c.append(evaluar_regresion(f"Ridge {alpha}", y_test, pred))

    lasso = Pipeline([
        ("scaler", StandardScaler()),
        ("model", Lasso(alpha=alpha, max_iter=10000))
    ])
    lasso.fit(X_train, y_train)
    pred = lasso.predict(X_test)
    resultados_c.append(evaluar_regresion(f"Lasso {alpha}", y_test, pred))

pd.DataFrame(resultados_c)

## Parte D — Árbol de regresión

Entrene árboles con:

- max_depth = 2
- max_depth = 5
- max_depth = 10
- max_depth = None

Compare RMSE train y RMSE test.

Responda:

1. ¿Qué profundidad tiene menor RMSE en test?
2. ¿Hay señales de overfitting?
3. ¿Qué pasa cuando no se limita la profundidad?
4. ¿Qué profundidad elegiría y por qué?

In [ ]:
# Parte D - Espacio de trabajo

prof_taller = [2, 5, 10, None]
resultados_d = []

for p in prof_taller:
    arbol = DecisionTreeRegressor(max_depth=p, random_state=42)
    arbol.fit(X_train, y_train)

    pred_train = arbol.predict(X_train)
    pred_test = arbol.predict(X_test)

    resultados_d.append({
        "max_depth": str(p),
        "RMSE_train": np.sqrt(mean_squared_error(y_train, pred_train)),
        "RMSE_test": np.sqrt(mean_squared_error(y_test, pred_test)),
        "R2_test": r2_score(y_test, pred_test)
    })

pd.DataFrame(resultados_d)

## Parte E — Conclusión profesional

Redacte una conclusión de mínimo 10 líneas respondiendo:

1. ¿Cuál modelo tuvo mejor desempeño?
2. ¿Cuál modelo fue más interpretable?
3. ¿Qué métrica usaría como principal y por qué?
4. ¿Qué evidencia encontró de overfitting?
5. ¿Qué modelo recomendaría para una primera implementación?
6. ¿Qué haría en una siguiente fase del proyecto?

La conclusión debe sonar como un informe técnico para un equipo de analítica.

# 25. Rúbrica sugerida del taller

| Criterio | Puntaje |
|---|---:|
| Comprensión de regresión vs clasificación | 0.6 |
| Regresión lineal e interpretación de coeficientes | 0.9 |
| Evaluación con MAE, RMSE y R² | 0.9 |
| Ridge y Lasso con análisis de alpha | 1.0 |
| Árbol de regresión y análisis de overfitting | 1.0 |
| Conclusión profesional | 0.6 |
| **Total** | **5.0** |

---

# Cierre de clase

Hoy cambiamos de clasificación a regresión.

Ya no preguntamos:

> ¿A qué clase pertenece?

Ahora preguntamos:

> ¿Qué valor numérico se espera?

Modelos vistos:

1. Regresión lineal.
2. Ridge.
3. Lasso.
4. Árbol de regresión.

---

# Siguiente clase sugerida

## Random Forest

Porque permite conectar:

- árboles individuales,
- overfitting,
- ensambles,
- mejora de estabilidad,
- clasificación y regresión.